# 00 · Start here: AI on the Edge

Welcome! In this course you will build a small **neural network that recognises hand gestures**. You will then make it **small and fast enough to run on a microcontroller**: an Arduino Nano 33 BLE Sense, a board the size of your thumb.

By the end of the course you can wave, punch or twist the board in your hand and it will tell you, by itself, which gesture you made. No laptop, no internet and no cloud are needed once it runs.

On the way you will learn the three most important techniques for shrinking neural networks:

| Technique | One-sentence idea | Notebook |
|---|---|---|
| **Quantization** | Store and compute with 8-bit integers instead of 32-bit decimal numbers. | 02 |
| **Pruning** (edge & node) | Remove the connections (edges) or neurons (nodes) the network does not really need. | 03 |
| **Knowledge distillation** | Let a big, accurate "teacher" network train a small "student" network. | 04 |

In notebook 05 you combine all three and **deploy the result on your board**.

> **Prerequisites.** Basic Python (variables, functions, lists, `for` loops). Some idea of what "machine learning" means is helpful, but every concept is explained where it is needed. You do *not* need to know PyTorch or C++.

## 1. What is "AI on the edge"?

Most AI you use runs in a **data centre**. Your phone records your voice, sends it over the internet to a big computer, and that computer sends the answer back. **Edge AI** (also called **TinyML** on microcontrollers) means running the AI model *on the device where the data is measured*: the "edge" of the network.

Why would you want that?

* **Latency**: no round trip over the network. A decision can take microseconds instead of hundreds of milliseconds.
* **Privacy**: the raw data (sound, motion, images) never leaves the device.
* **Energy**: sending data by radio often costs far more energy than processing it locally. A coin battery can power a microcontroller for months.
* **No connectivity needed**: it works in a forest, a factory or a tunnel.
* **Cost**: a microcontroller costs a few euros and needs no server.

The price you pay is that the hardware is *tiny*. Compare:

| | Typical laptop | Arduino Nano 33 BLE Sense |
|---|---|---|
| Processor | 4-16 cores @ ~3 GHz | 1 core (Arm Cortex-M4F) @ **64 MHz** |
| Working memory (RAM) | 8-32 GB | **256 KB** (~65 000x less than 16 GB) |
| Storage | 512 GB SSD | **1 MB** flash memory |
| Power | 20-60 W | a few **milli**watts |
| Operating system | Windows / macOS / Linux | none (or a tiny real-time OS) |

A modern image-recognition network easily needs hundreds of megabytes. It would not even fit on the board, let alone run fast. That is why we need the techniques of this course.

## 2. Meet the board

The **Arduino Nano 33 BLE Sense** is built around the **nRF52840** chip by Nordic Semiconductor:

* **CPU**: Arm **Cortex-M4F** at 64 MHz. The "F" means it has a hardware **floating-point unit (FPU)**, so it can compute with 32-bit decimal numbers quickly. Many cheaper microcontrollers cannot.
* **Flash memory (1 MB)** is where your *program* lives, including the network's **weights**. It is kept when the power is off and is read-only while the program runs.
* **RAM (256 KB)** holds the data that changes while the program runs: sensor readings, the intermediate results of the network ("activations"), and so on.
* **Bluetooth Low Energy** radio (we will not use it, but you could send predictions to a phone).

And it has many **sensors** on board. Two hardware revisions exist:

| Sensor | Nano 33 BLE Sense (Rev1) | Nano 33 BLE Sense **Rev2** |
|---|---|---|
| **Motion (IMU)**: accelerometer + gyroscope (+ magnetometer) | LSM9DS1 | BMI270 + BMM150 |
| Pressure / temperature | LPS22HB | LPS22HB |
| Humidity / temperature | HTS221 | HS3003 |
| Light, colour, proximity, gesture | APDS9960 | APDS9960 |
| Microphone | MP34DT05 | MP34DT06JTR |

We use the **IMU** (*Inertial Measurement Unit*): its accelerometer and gyroscope are what let the board "feel" a gesture. The code in this course supports both revisions; you only have to tell it which one you have (see section 5).

## 3. The project: gesture recognition

You hold the board flat in your hand, components facing up, with the USB cable pointing towards your wrist. Then you perform one of five gestures:

| Gesture | How to do it |
|---|---|
| **punch** | A quick jab forward (away from you) and back. |
| **lift** | Raise your hand ~20 cm and bring it back down. |
| **wave** | Move your hand left-right-left (1-3 times). |
| **circle** | Draw one vertical circle in the air in front of you. |
| **twist** | Turn your wrist (like turning a door knob) and back. |

The board notices that it starts moving and records **1 second** of motion data: 50 readings, each with 6 numbers. A neural network turns these **300 numbers** into a decision: *which of the 5 gestures was it?*

### The workflow of every edge-AI project

```
  collect data  ->  train a model  ->  compress it  ->  convert to C++  ->  run on the device
  (notebook 01)    (notebook 01)    (notebooks 02-04)   (notebook 05)      (notebook 05)
```

Training happens on your laptop: it needs lots of computation and many examples. Only the finished, compressed model goes to the board. Running a trained model on new data is called **inference**.

## 4. Course map

| Notebook | What you do | Board needed? |
|---|---|---|
| **00_start_here** (this one) | Overview, set-up check. | no |
| **01_data_and_first_model** | Sensor data, neural-network basics, train the *baseline* model. Optionally record your own gestures. | optional |
| **02_quantization** | float32 → int8. Post-training quantization and quantization-aware training. | no |
| **03_pruning** | Edge pruning (remove weights) and node pruning (remove neurons). | no |
| **04_distillation** | A big teacher trains a small student. | no |
| **05_deploy_to_arduino** | Combine everything, generate C++ code, run it on the board. | **yes** |

Every notebook can be run on its own: it re-creates the data and re-trains what it needs (this takes seconds). Small helper functions live in the `aiedge/` folder. You are encouraged to open those files: they are short and commented. The important code (the quantization formulas, the pruning steps and so on) is written out *in the notebooks*.

Each notebook ends with **exercises**. Do at least the ones marked ⭐.

## 5. Setting up

### 5a. Python (for notebooks 00-05)

In a terminal, in the project folder:

```bash
uv sync                      # installs Python packages into .venv (once)
uv run jupyter lab           # opens Jupyter in your browser
```

No `uv`? Install it from <https://docs.astral.sh/uv/>, or use `pip install torch numpy matplotlib jupyterlab pyserial` in a virtual environment.

**Jupyter basics**: a notebook is a list of *cells*. Click a code cell and press **Shift + Enter** to run it. Run cells **from top to bottom**: later cells use variables created by earlier ones. If something behaves strangely, use *Kernel → Restart Kernel and Run All Cells*.

Run the next cell. It should print the versions and "Everything is ready".

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))   # lets us `import aiedge` from the project folder

import numpy as np
import torch
import matplotlib

from aiedge import config as C

print("Python    ", sys.version.split()[0])
print("PyTorch   ", torch.__version__)
print("NumPy     ", np.__version__)
print("Matplotlib", matplotlib.__version__)
print()
print(f"Gestures: {C.GESTURES}")
print(f"One gesture = {C.WINDOW_SAMPLES} readings x {C.N_CHANNELS} channels = {C.N_FEATURES} numbers")
print("Everything is ready ✅")

### 5b. Arduino (needed for notebook 05, and optionally for recording data in notebook 01)

1. Install the **Arduino IDE 2** from <https://www.arduino.cc/en/software>.
2. *Tools → Board → Boards Manager…*, search **"Arduino Mbed OS Nano Boards"** and install it. (This also installs the USB drivers on Windows.)
3. *Tools → Manage Libraries…*, install the IMU library for your board:
   * Rev1: **Arduino_LSM9DS1**
   * Rev2: **Arduino_BMI270_BMM150**

   Not sure which revision you have? Look at the box or the shop page. Or simply try: the sketches print `IMU not found. Is BOARD_REVISION set correctly?` if you picked the wrong one.
4. Plug in the board, select *Tools → Board → Arduino Mbed OS Nano Boards → Arduino Nano 33 BLE* and the right *Port*.
5. Test: *File → Examples → 01.Basics → Blink*, then **Upload** (→ button). The orange LED should blink.

> **Upload problems?** Double-press the small reset button on the board. The orange LED starts "breathing" (bootloader mode). Select the port again and upload.

Prefer the command line? [`arduino-cli`](https://arduino.github.io/arduino-cli/) works too. Notebook 05 shows the commands.

## 6. A word about the data

Recording thousands of gestures takes time. So this course comes with a **gesture simulator** (`aiedge/data.py`) that generates realistic sensor data from physics: the motion of a hand, gravity, the tilt of the board, sensor noise, and the differences between people. All notebooks run on this **synthetic** data out of the box, without a board.

The simulator is a model of reality, not reality itself. Your real board's sensor axes, your personal way of waving, and so on will differ a bit. So the best model for *your* board is trained on data *you* recorded with it. Notebook 01 shows how (about 15 minutes of recording), and notebook 05 can train on your recordings with a single switch.

Ready? Open **`01_data_and_first_model.ipynb`**.